# RIEMPIRE DATASET DINOS

### Si cerca di andare a riempire le colonne che interessano per l'analisi con altri csv

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("dinosauri_clean.csv")

# Crea una copia di sicurezza indipendente
df_rivisitato = df.copy()

In [2]:
# Lista di tutte le varianti testuali che indicano un dato mancante nel tuo CSV
valori_da_annullare = ["Sconosciuto", "Sconosciuta", "NO_ORDER_SPECIFIED", "NO_FAMILY_SPECIFIED"]

colonne_da_sistemare = ['order', 'family', 'genus', 'state', 'formation']

In [3]:
# Sostituisci il testo con il vero valore nullo di Numpy (np.nan)
for col in colonne_da_sistemare:
    if col in df_rivisitato.columns:
        df_rivisitato[col] = df_rivisitato[col].replace(valori_da_annullare, np.nan)

In [4]:
# Carica i due nuovi dataset del tuo compagno
df_2 = pd.read_csv("dinos_2.csv")
df_3 = pd.read_csv("dinos_3.csv")

In [5]:
# Creiamo colonne pulite in minuscolo in tutti i dataset per fare il match perfetto
df_rivisitato['genus_lower'] = df_rivisitato['genus'].astype(str).str.lower().str.strip()
df_2['name_lower'] = df_2['name'].astype(str).str.lower().str.strip()
df_3['name_lower'] = df_3['name'].astype(str).str.lower().str.strip()

In [ ]:
# Recupero dati da df_2

# 1. Creiamo la mappa per le famiglie e riempiamo i NaN
mappa_famiglia = df_2.dropna(subset=['family']).drop_duplicates(subset=['name_lower']).set_index('name_lower')['family']
df_rivisitato['family'] = df_rivisitato['family'].fillna(df_rivisitato['genus_lower'].map(mappa_famiglia))

# 2. Creiamo la mappa per gli stati (regioni) e riempiamo i NaN
mappa_stato = df_2.dropna(subset=['region']).drop_duplicates(subset=['name_lower']).set_index('name_lower')['region']
df_rivisitato['state'] = df_rivisitato['state'].fillna(df_rivisitato['genus_lower'].map(mappa_stato))

In [ ]:
# Funzione per estrarre l'Ordine da df_3 e mappatura

def estrai_ordine(taxonomy_str):
    if pd.isna(taxonomy_str):
        return np.nan
    
    testo = str(taxonomy_str).lower()
    
    # Cerchiamo le parole chiave tassonomiche all'interno della stringa
    if 'theropoda' in testo:
        return 'Theropoda'
    elif 'sauropodomorpha' in testo or 'sauropoda' in testo:
        return 'Sauropodomorpha'
    elif 'ornithopoda' in testo:
        return 'Ornithopoda'
    elif 'ceratopsia' in testo:
        return 'Ceratopsia'
    elif 'stegosauria' in testo:
        return 'Stegosauria'
    elif 'ankylosauria' in testo:
        return 'Ankylosauria'
    elif 'saurischia' in testo:
        return 'Saurischia'
    elif 'ornithischia' in testo:
        return 'Ornithischia'
    
    return np.nan

# 1. Applichiamo la funzione su df_3 per creare una colonna ordine pulita
df_3['order_extracted'] = df_3['taxonomy'].apply(estrai_ordine)

# 2. Creiamo la mappa basata sul nome del dinosauro
mappa_ordine = df_3.dropna(subset=['order_extracted']).drop_duplicates(subset=['name_lower']).set_index('name_lower')['order_extracted']

# 3. Riempiamo i NaN nella colonna 'order' del tuo df_rivisitato
df_rivisitato['order'] = df_rivisitato['order'].fillna(df_rivisitato['genus_lower'].map(mappa_ordine))

In [8]:
df_rivisitato.to_csv("dinosauri_integrazione.csv", index=False)

In [10]:
# 1. Contiamo quanti dati mancanti c'erano nel file ORIGINALE (prima della cura)
valori_da_annullare = ["Sconosciuto", "Sconosciuta", "NO_ORDER_SPECIFIED", "NO_FAMILY_SPECIFIED"]
df = pd.read_csv("dinosauri_clean.csv")
for col in ['order', 'family', 'state']:
    df[col] = df[col].replace(valori_da_annullare, np.nan)

# 2. Stampiamo il confronto "Prima vs Dopo"
print("=== CONFRONTO DATI MANCANTI ===")
print("\n[PRIMA] Nel dataset originale:")
print(df[['order', 'family', 'state']].isnull().sum())

print("\n[DOPO] Nel tuo dataset integrato:")
print(df_rivisitato[['order', 'family', 'state']].isnull().sum())

=== CONFRONTO DATI MANCANTI ===

[PRIMA] Nel dataset originale:
order     21313
family    10444
state       570
dtype: int64

[DOPO] Nel tuo dataset integrato:
order     17365
family     9879
state       546
dtype: int64


### Domande da poter aggiungere 

1. La crisi delle piante ha causato la fame nel Mesozoico?
Spiegazione dei Rettiliani: "Prima dell'asteroide stavamo già morendo di fame? Vogliamo vedere se il calo della vegetazione (cibo) in certe ere ha trascinato al declino anche noi rettili erbivori, firmando la nostra condanna in anticipo."

2. Quali specie erano già "con le valigie pronte" prima del K-Pg?
Spiegazione dei Rettiliani: "Eravamo davvero all'apice del successo o stavamo già scomparendo? Dobbiamo capire quale gruppo (es. i grandi erbivori o i carnivori) stava subendo il crollo di biodiversità più drastico nell'ultima era prima dell'impatto."

3. C'erano dei "Rifugi Sicuri" dove l'estinzione è stata peggiore?
Spiegazione dei Rettiliani: "Se ci fossimo spostati in massa, ci saremmo salvati? Analizziamo la mappa per vedere dove la coesistenza tra piante e dinosauri era massima e se l'estinzione ha colpito più duramente le zone isolate o quelle sovraffollate."
